# Ch9 LoRA 与量化 教案

**课程名称：** LoRA 与量化：面向消费级 GPU 的高效微调

**预计总时长：** 85 分钟

**源文件：** `Ch9_LoRA_Quantization/Ch9_LoRA_Quantization.ipynb`（共37 个 Cell，Cell 0-36）

---

## 时间表

| 时间段 | 内容 | Cell 范围 | 时长 |
|:---|:---|:---|:---|
| 0-5 min | 开场 + 环境准备 | Cell 0-5 | 5 min |
| 5-20 min | LoRA 理论：低秩分解 + 前向传播公式 + 参数压缩 | Cell 6-8 | 15 min |
| 20-35 min | 动手练习：从零实现 LoRA 前向传播 | Cell 9-15 | 15 min |
| 35-40 min | **休息 + 回顾** | -- | 5 min |
| 40-55 min | 量化基础：INT8/INT4 对称量化 + 可视化 | Cell 16-19 | 15 min |
| 55-70 min | PEFT 实战：GPT-2 LoRA 微调 | Cell 20-28 | 15 min |
| 70-80 min | QLoRA：4-bit 量化 + LoRA 组合方案 | Cell 29-32 | 10 min |
| 80-85 min | 总结 + 延伸阅读 + 课后练习 | Cell 33-36 | 5 min |

---

## 课前准备

- [ ] 确认 GPU 环境就绪（RTX 5080 / 17.1GB 或同级别）
- [ ] 安装 `transformers`, `peft`, `datasets`（Cell 5 会自动安装）
- [ ] 确认 `uer/gpt2-chinese-cluecorpussmall` 模型可下载
- [ ] 预跑 Cell 4-5，确保无报错
- [ ] 准备白板或投影笔，用于画矩阵分解示意图

## 关键数据速查

| 指标 | 数值 |
|:---|:---|
| GPU | RTX 5080, 17.1GB |
| 模型总参数量 | 118,295,040 (0.47GB) |
| LoRA 可训练参数 | 1,622,016 (1.35%), 6.49MB |
| LoRA 配置 | rank=16, alpha=32, target=c_attn+c_proj |
| 压缩比 | rank=4→512x, rank=8→256x, rank=16→128x |
| 量化效果 | INT8: 4x压缩, err=0.009; INT4: 8x压缩, err=0.165 |
| 训练 | 6 epochs, loss 0.0619→0.0000, 75.9s |
| QLoRA 显存 | 7B→3.6GB, 13B→6.6GB, 30B→15.1GB |

---

## 第一段：开场与环境准备（Cell 0-5）

📍 运行 Cell 0-1（Markdown 导读）、Cell 3（环境准备标题）、Cell 4-5（代码）

⏱ 时间分配：5 分钟

🎯 本段目标
- 建立学习动机：全量微调 7B 模型需要 56GB 显存，消费级 GPU 根本装不下
- 确认环境就绪（PyTorch、PEFT、GPU 信息）
- 从 Ch8 SFT 自然过渡到“如何降低微调成本”

🗣 讲课话术

> 大家好！上一章我们学了 SFT 指令微调，知道了怎么用 (instruction, response) 数据对来训练模型。但有个现实问题——如果模型有 70 亿参数，全量微调需要多少显存？
>
> 我们来算一下。模型参数 FP16 占 14GB，梯度也是 14GB，Adam 优化器要存两个状态又是 28GB。加起来就是 **56GB**。大家手里的 GPU 多大？（等回应）对，一般 8-24GB。差了好几倍！
>
> 所以今天的核心问题是：**能不能只训练一小部分参数，达到接近全量微调的效果？** 答案就是 LoRA——Low-Rank Adaptation。
>
> 先运行 Cell 4，确认环境。（运行 Cell 4）大家看到设备和 PyTorch 版本了吗？再运行 Cell 5 安装 PEFT 库。这个库是 Hugging Face 出的参数高效微调工具箱。

👀 输出要点
- Cell 4：`device: cuda`，PyTorch 版本，GPU 型号 RTX 5080 / 17.1GB
- Cell 5：PEFT 安装成功确认

❓ Q&A 预案
- **Q：没有 GPU 怎么办？** A：本章的 LoRA 手写实现和量化部分都可以在 CPU 上跑。PEFT 微调部分，CPU 会比较慢但也能演示。
- **Q：为什么不用 DeepSpeed/FSDP？** A：那是分布式方案，需要多卡。LoRA 的目标是让单卡也能微调大模型。

➡️ 转场

> 环境就绪了。现在我们来看 LoRA 的核心思想——低秩矩阵分解。这个概念用一句话就能概括：**大矩阵的变化量可以用两个小矩阵的乘积来近似**。

---

## 第二段：LoRA 理论——低秩分解与前向传播（Cell 6-8）

📍 浏览 Cell 6（前向传播公式）、Cell 7（理论深化：内在维度假说）、运行 Cell 8（参数量可视化）

⏱ 时间分配：15 分钟

🎯 本段目标
- 掌握 LoRA 前向传播公式：$y = Wx + BAx \cdot \frac{\alpha}{r}$
- 理解 B 零初始化的意义：训练开始时 ΔW = 0，保持原始模型行为
- 通过参数量对比建立直觉：rank=16 时压缩 128 倍

🗣 讲课话术

> 来看 Cell 6，LoRA 的核心公式就一行：$y = Wx + BAx \cdot \frac{\alpha}{r}$
>
> 左边 $Wx$ 是原始的线性层，这部分权重**冻结不动**。右边是 LoRA 增量：先把输入 $x$ 通过矩阵 $A$ 从 $d_{in}$ 维降到 $r$ 维（下投影），再通过矩阵 $B$ 从 $r$ 维升回 $d_{out}$ 维（上投影），最后乘以缩放因子 $\alpha/r$。
>
> 这里 $r$ 就是秩（rank），通常取 4、8、16 这样很小的数。原始权重矩阵是 $d_{in} \times d_{out}$，比如 768×768 = 589,824 个参数。而 LoRA 只需要 $A$（$r \times d_{in}$ = 16×768 = 12,288）加 $B$（$d_{out} \times r$ = 768×16 = 12,288），总共 24,576 个参数。**压缩了 24 倍！** 如果 $r=4$，压缩比更是高达 512 倍。
>
> 一个关键设计：**B 初始化为零**。这意味着训练开始时 $\Delta W = BA = 0 \times A = 0$，模型输出和原始模型完全一致。LoRA 从“不改变”开始，逐步学习微调方向。这比随机初始化稳定得多。
>
> 为什么低秩近似有效？Cell 7 讲了“内在维度假说”。简单说：微调时权重的变化 $\Delta W$ 看起来有几百万个参数在动，但实际上这些变化集中在一个低维子空间里。对 $\Delta W$ 做 SVD 分解，你会发现只有前几个奇异值是显著的。LoRA 就是直接在这个低维空间里学习。
>
> 现在运行 Cell 8 看参数量对比。（运行 Cell 8）

👀 输出要点
- Cell 8 输出柱状图：rank=4/8/16/32 对应的参数量对比
- 压缩比：rank=4→512x, rank=8→256x, rank=16→128x, rank=32→64x
- 直观感受：所有 LoRA 参数加起来只有原始权重的一个零头

❓ Q&A 预案
- **Q：α 和 r 的关系是什么？** A：α/r 是缩放因子。通常 α=2r（比如 r=16 时 α=32），这样 scaling=2。α 越大，LoRA 增量对输出的影响越大。实践中 α 和 r 一起作为超参数调优。
- **Q：为什么不直接用 SVD 分解原始权重？** A：原始权重不是低秩的，低秩的是微调时的**变化量** ΔW。LoRA 不是去压缩原始模型，而是用低秩矩阵去表示微调学到的东西。
- **Q：LoRA 的精度损失大吗？** A：Hu et al. 2021 的论文显示，rank=8 的 LoRA 在多数 NLP 任务上与全量微调效果持平，甚至有时更好（因为低秩约束起到了正则化效果）。

➡️ 转场

> 理论讲完了，现在动手！我们要从零实现 LoRA 的前向传播。Cell 9-15 是一个动手练习区，大家先自己试，再看参考答案。

---

## 第三段：动手练习——从零实现 LoRA 前向传播（Cell 9-15）

📍 浏览 Cell 9（标题）、Cell 10（题目说明）、运行 Cell 11（练习框架）、Cell 12（验证）、Cell 13-15（参考实现 + 测试）

⏱ 时间分配：15 分钟（5 分钟读题 + 5 分钟做题 + 5 分钟讲解）

🎯 本段目标
- 学生亲手实现 `y = Wx + BAx * scaling`
- 理解 `F.linear(x, A)` = `x @ A^T` 的含义
- 通过验证代码确认实现正确

🗣 讲课话术

> 来看 Cell 10 的题目。核心就是补全 `lora_forward` 函数。公式是 `y = Wx + BAx * scaling`。
>
> 几个提示：
> 1. `F.linear(x, W)` 等价于 `x @ W^T`，这是 PyTorch 的标准线性层操作
> 2. 计算 LoRA 增量的步骤是：先 `F.linear(x, A)` 得到低维表示，再 `F.linear(result, B)` 升回高维，最后乘 `scaling`
> 3. 最终输出是原始输出加上 LoRA 增量
>
> 给大家 5 分钟时间，请运行 Cell 11 并补全 `# TODO` 部分。

**提示节奏（根据进度逐步释放）：**

| 时间 | 提示 |
|:---|:---|
| 1 分钟后 | 第一步：`base_output = F.linear(x, W, bias)` 得到原始输出 |
| 2 分钟后 | 第二步：`after_A = F.linear(x, A)` 把输入降维到 rank 维 |
| 3 分钟后 | 第三步：`after_B = F.linear(after_A, B)` 把低维表示升回原始维度 |
| 4 分钟后 | 第四步：`return base_output + after_B * scaling` 加上缩放后的增量 |

> （5 分钟后）时间到，我们来看参考答案。运行 Cell 12 验证。（运行 Cell 12）
>
> 看到 ✓ 通过 了吗？核心就四行代码。注意这里没有任何神秘的东西——LoRA 本质就是在原始线性层旁边并联了一条低秩旁路。
>
> 现在运行 Cell 13-15，看完整的 LoRALinear 类实现和测试。（依次运行）

👀 输出要点
- Cell 12：验证通过提示
- Cell 14：LoRA Linear 测试——输出维度正确、LoRA 增量非零、冻结参数不在梯度计算图中
- Cell 15：梯度流验证——只有 A 和 B 有梯度，W 没有

**常见错误：**
- 忘记乘 `scaling`：输出数值不对但维度正确
- `F.linear` 参数顺序写反：`F.linear(A, x)` 会报维度错误
- 把 LoRA 增量减去而不是加上：符号错误

❓ Q&A 预案
- **Q：为什么用 `F.linear` 而不是直接矩阵乘法？** A：`F.linear(x, W)` 自动处理 batch 维度和转置，等价于 `x @ W.T`。直接写 `@` 也可以，但 `F.linear` 更规范。
- **Q：训练时 W 真的不更新吗？** A：对，Cell 15 的梯度验证会证明这一点。W 的 `requires_grad=False`，反向传播时梯度只流过 A 和 B。

➡️ 转场

> 做完练习，大家对 LoRA 的实现应该很清楚了。我们休息 5 分钟，回来看量化——另一个减少显存的关键技术。

---

## ☕ 休息 + 回顾（5 分钟）

**已完成：**
- ✅ LoRA 理论：低秩分解、前向传播公式、参数压缩比
- ✅ 动手实现 LoRA 前向传播

**回顾问题（休息时可以思考）：**
1. LoRA 的前向传播公式是什么？
2. 为什么 B 矩阵要初始化为零？
3. rank=16 时，768×768 的权重矩阵压缩了多少倍？

**下半场预告：**
- 量化基础（INT8/INT4）
- PEFT 实战微调
- QLoRA 组合方案

---

## 第四段：量化基础——INT8/INT4 对称量化（Cell 16-19）

📍 浏览 Cell 16-17（量化理论 Markdown）、运行 Cell 18（量化实现代码）、Cell 19（量化效果可视化）

⏱ 时间分配：15 分钟

🎯 本段目标
- 理解量化的核心思想：用更少的 bit 表示权重，换取显存节省
- 掌握对称量化公式：scale = max(|x|) / (2^(b-1) - 1)
- 通过数值例子和可视化建立直觉：INT8 误差极小，INT4 有可见误差

🗣 讲课话术

> 欢迎回来！LoRA 解决的是“训练哪些参数”的问题，量化解决的是“参数占多少空间”的问题。
>
> 看 Cell 16 的表格。FP32 每个参数 4 字节，7B 模型就是 28GB。FP16 减半到 14GB。INT8 只要 1 字节，7GB。INT4 更是只要 0.5 字节，3.5GB。
>
> 量化怎么做的？来看一个具体例子。假设权重值是 0.7532（FP32，32 bit）。INT8 量化过程：
> 1. 找到这一层权重的最大绝对值，比如 1.0
> 2. 计算 scale = 1.0 / 127 = 0.00787
> 3. 量化：round(0.7532 / 0.00787) = round(95.7) = 96
> 4. 存储 96 这个整数，只占 8 bit
>
> 推理时反量化：96 × 0.00787 = 0.7555。和原始值 0.7532 的误差只有 0.003。**显存省了 75%，精度几乎没损失！**
>
> 但 INT4 就不同了。4 bit 只能表示 -8 到 7 共 16 个值，量化步长大了很多，误差也大了。
>
> 为什么量化有效？Cell 17 讲了三个原因：过参数化带来冗余、训练好的模型在损失面的平坦区域、量化误差在多层间统计平均。
>
> 现在运行 Cell 18-19 看实际效果。（运行 Cell 18, 19）

👀 输出要点
- Cell 18：对称量化函数实现——scale 计算、round、clamp
- Cell 19 图表：
  - INT8 量化：压缩 4x，误差 0.009（几乎看不出差异）
  - INT4 量化：压缩 8x，误差 0.165（权重分布有明显阶梯化）
  - 直方图对比：INT8 分布与原始几乎重合，INT4 有可见偏移

❓ Q&A 预案
- **Q：INT4 误差这么大，模型还能用吗？** A：单层看误差不小，但实践中配合一些技巧（NF4 分布适配、分组量化、双重量化），INT4 模型在推理上的质量损失出人意料地小。这就是 GPTQ、AWQ 等量化方法的贡献。
- **Q：对称量化和非对称量化有什么区别？** A：对称量化以 0 为中心，正负对称映射；非对称量化允许 zero_point 不为 0，能更好地处理权重分布不对称的情况。Cell 17 有详细对比。
- **Q：量化是在训练时做还是推理时做？** A：传统量化是训练后做（Post-Training Quantization）。QLoRA 是训练时用量化权重，但训练的是 LoRA 参数而不是量化参数。

➡️ 转场

> 理解了量化基础，我们来看实战！接下来用 PEFT 库在真实的 GPT-2 模型上做 LoRA 微调。

---

## 第五段：PEFT 实战——GPT-2 LoRA 微调（Cell 20-28）

📍 运行 Cell 21（加载模型）、Cell 22（LoRA 配置）、Cell 23（创建 LoRA 模型 + 参数统计）、Cell 24（可视化）、Cell 25（准备数据）、Cell 26（训练循环）、Cell 27（Loss 曲线）、Cell 28（生成测试）

⏱ 时间分配：15 分钟

🎯 本段目标
- 用 PEFT 库 3 行代码给 GPT-2 加上 LoRA
- 直观感受参数量差异：118M 总参数 vs 1.62M 可训练参数（1.35%）
- 观察训练过程：6 epochs, loss 从 0.0619 降到 0.0000, 75.9 秒

🗣 讲课话术

> Cell 20 说了，我们现在要在 GPT-2 中文版上做实际微调。这个模型有 1.18 亿参数，0.47GB。
>
> 运行 Cell 21，加载模型和 tokenizer。（运行 Cell 21）模型名是 `uer/gpt2-chinese-cluecorpussmall`，中文语料预训练的 GPT-2。
>
> 现在看 Cell 22 的 LoRA 配置——这是最关键的几行代码：
> ```python
> LoraConfig(
>     r=16,           # rank=16
>     lora_alpha=32,  # alpha=32, scaling=2
>     target_modules=["c_attn", "c_proj"],  # 只在注意力层加 LoRA
>     lora_dropout=0.05,
>     bias="none"
> )
> ```
>
> `r=16` 就是我们讲的秩。`target_modules` 指定在哪些层加 LoRA——我们选了注意力层的 `c_attn`（Q/K/V 投影）和 `c_proj`（输出投影）。这是最常见的配置，因为注意力层是 Transformer 的核心。
>
> 运行 Cell 23。（运行 Cell 23）看到了吗？**总参数 118,295,040，可训练参数 1,622,016，只有 1.35%！** 这就是 LoRA 的威力——只训练 6.49MB 的参数，就能微调一个 0.47GB 的模型。
>
> Cell 24 的可视化更直观。（运行 Cell 24）柱状图对比了原始参数量和 LoRA 参数量，差距一目了然。
>
> Cell 25-26 是训练部分。我们用中文对联数据做演示。（运行 Cell 25, 26）
>
> 看训练输出：6 个 epoch，loss 从 0.0619 一路降到 0.0000，总共 75.9 秒。在 RTX 5080 上非常快。
>
> 运行 Cell 27 看 loss 曲线。（运行 Cell 27）非常漂亮的下降曲线，说明 LoRA 的训练效率很高。
>
> 最后 Cell 28，用微调后的模型生成文本。（运行 Cell 28）

👀 输出要点
- Cell 21：模型加载成功
- Cell 23：`trainable params: 1,622,016 || all params: 118,295,040 || trainable%: 1.35%`，LoRA 参数体积 6.49MB
- Cell 24：参数量对比柱状图
- Cell 26：训练日志——6 epochs, loss 0.0619→0.0000, 75.9s
- Cell 27：Loss 曲线图
- Cell 28：微调后生成效果

❓ Q&A 预案
- **Q：为什么只对 c_attn 和 c_proj 加 LoRA？** A：实验表明注意力层的权重对微调效果贡献最大。也可以加在 MLP 层（c_fc, c_proj），但参数量会增加且边际收益递减。
- **Q：6 epochs 就过拟合了吗？loss 降到 0？** A：对，在小数据集上 loss 降到 0 确实说明过拟合了。真实场景下数据量大得多，loss 不会降到 0。这里是演示目的。
- **Q：1.35% 的参数能学到什么？** A：LoRA 学习的是任务特定的“微调方向”，不是重新学习语言能力。预训练已经给了模型强大的语言基础，LoRA 只需要在特定方向上做微小调整。

➡️ 转场

> 我们用 PEFT 做了完整的 LoRA 微调。但如果模型更大呢？7B、13B？光加载 FP16 权重就要 14-26GB。这时候就需要 QLoRA——把量化和 LoRA 结合起来。

---

## 第六段：QLoRA——量化与 LoRA 的结合（Cell 29-32）

📍 浏览 Cell 29（QLoRA 理论对比表）、运行 Cell 30（显存对比可视化）、浏览 Cell 31（QLoRA 配置代码示例）、Cell 32（可选 QLoRA demo）

⏱ 时间分配：10 分钟

🎯 本段目标
- 理解 QLoRA = 4-bit 量化（冻结基座）+ LoRA（训练适配器）
- 记住关键数据：7B→3.6GB, 13B→6.6GB, 30B→15.1GB
- 了解 NF4（NormalFloat4）和双重量化的核心思想

🗣 讲课话术

> Cell 29 有一张关键对比表。普通 LoRA 的基座权重是 FP16，7B 模型需要约 14GB 光是加载权重。QLoRA 把基座权重量化到 INT4（准确说是 NF4），7B 模型只需要 **3.5GB** 加载权重，再加 LoRA 参数也就 **3.6GB** 左右。
>
> 运行 Cell 30 看显存对比。（运行 Cell 30）
>
> 看这张图！7B 模型：全量微调 56GB，LoRA (FP16) 14.2GB，QLoRA 只要 **3.6GB**。13B 模型 QLoRA 只要 6.6GB。**这意味着一张 8GB 的 RTX 4060 就能微调 7B 模型！**
>
> QLoRA 有三个关键创新：
> 1. **NF4 量化**：不是普通的 INT4，而是根据神经网络权重的正态分布特点设计的 4-bit 格式，量化误差比普通 INT4 小很多
> 2. **双重量化**：连量化的 scale 参数本身也做一次量化，进一步节省空间
> 3. **分页优化器**：当 GPU 显存不够时，自动把优化器状态卸载到 CPU 内存
>
> Cell 31 展示了 QLoRA 的代码配置，核心就是加一个 `BitsAndBytesConfig`。如果环境有 `bitsandbytes` 库，Cell 32 可以实际运行。

👀 输出要点
- Cell 30：显存对比柱状图
  - 7B: Full=56GB, LoRA=14.2GB, QLoRA=3.6GB
  - 13B: Full=104GB, LoRA=26.2GB, QLoRA=6.6GB
  - 30B: Full=240GB, LoRA=60.2GB, QLoRA=15.1GB
- Cell 31：QLoRA 配置代码（BitsAndBytesConfig + LoraConfig）

❓ Q&A 预案
- **Q：QLoRA 训练速度比普通 LoRA 慢多少？** A：大约慢 20-30%，因为每次前向传播都需要把 INT4 权重反量化到 FP16 再计算。但显存节省带来的收益远大于速度损失。
- **Q：QLoRA 微调出来的模型质量和全量微调差多少？** A：Dettmers et al. 2023 的论文显示，QLoRA 在 MMLU 等基准上与全量微调的 ChatGPT 水平模型差距很小。在 Guanaco 数据集上甚至达到了 ChatGPT 99.3% 的水平。
- **Q：bitsandbytes 在 Windows 上能用吗？** A：最新版已经支持 Windows，但安装可能需要注意 CUDA 版本兼容性。Linux 上最稳定。

➡️ 转场

> QLoRA 是目前消费级 GPU 微调大模型的主流方案。我们来做最后的总结。

---

## 第七段：总结与课后练习（Cell 33-36）

📍 浏览 Cell 33（总结）、Cell 34（延伸阅读）、Cell 35-36（课后练习）

⏱ 时间分配：5 分钟

🎯 本段目标
- 串联全章核心概念：LoRA → 量化 → QLoRA
- 指明课后练习方向
- 连接到下一章 DPO

🗣 讲课话术

> 我们来回顾今天学的内容。Cell 33 有一张核心概念图谱。
>
> 三条线索串起全章：
> 1. **LoRA**：冻结原始权重，只训练低秩旁路 BA。rank=16 压缩 128 倍，1.35% 的参数就能微调。
> 2. **量化**：用更少的 bit 表示权重。INT8 压缩 4 倍几乎无损，INT4 压缩 8 倍有少量误差。
> 3. **QLoRA**：两者结合。4-bit 量化冻结基座 + LoRA 训练适配器。7B 模型只需 3.6GB 显存。
>
> 课后练习在 Cell 35：
> - 尝试不同 rank（4、8、16、32）对比效果
> - 把 LoRA 加到不同的层看差异
> - 实现权重合并（把 LoRA 参数融合回原始权重）
> - 如果有 bitsandbytes，在更大模型上试 QLoRA
>
> 下一章是 **DPO 偏好优化**——用偏好数据直接优化模型输出风格，不需要训练奖励模型，比 RLHF 更简洁。

❓ Q&A 预案
- **Q：工业界主要用 LoRA 还是 QLoRA？** A：看场景。有多卡集群的公司会用全量微调或普通 LoRA。个人开发者和小团队几乎都用 QLoRA，因为一张消费级 GPU 就够了。
- **Q：LoRA 和 Adapter Tuning、Prefix Tuning 有什么区别？** A：都是参数高效微调方法。Adapter 在每层中间插入小型 MLP，Prefix Tuning 在输入前拼接可学习的 prefix token。LoRA 直接修改现有权重矩阵，不改变模型结构，推理时可以合并回去无额外开销，这是它的最大优势。

---

## 附录

### 时间表速查

| 时间段 | 内容 | 时长 |
|:---|:---|:---|
| 0-5 min | 开场 + 环境准备 | 5 min |
| 5-20 min | LoRA 理论 | 15 min |
| 20-35 min | 动手练习：LoRA 前向传播 | 15 min |
| 35-40 min | 休息 | 5 min |
| 40-55 min | 量化基础 | 15 min |
| 55-70 min | PEFT 实战微调 | 15 min |
| 70-80 min | QLoRA | 10 min |
| 80-85 min | 总结 | 5 min |

### 关键数据速查

| 指标 | 数值 |
|:---|:---|
| 模型 | uer/gpt2-chinese-cluecorpussmall |
| 总参数量 | 118,295,040 (0.47GB) |
| LoRA 可训练参数 | 1,622,016 (1.35%), 6.49MB |
| LoRA 配置 | r=16, α=32, target=c_attn+c_proj |
| 压缩比 | r=4→512x, r=8→256x, r=16→128x |
| INT8 量化 | 4x 压缩, err=0.009 |
| INT4 量化 | 8x 压缩, err=0.165 |
| 训练 | 6 epochs, loss 0.0619→0.0000, 75.9s |
| QLoRA 显存 | 7B→3.6GB, 13B→6.6GB, 30B→15.1GB |

### 应急预案

| 问题 | 解决方案 |
|:---|:---|
| GPU 显存不足 | 减小 batch_size；用 CPU 演示手写 LoRA 部分 |
| PEFT 安装失败 | `pip install peft --no-deps`，或用预装环境 |
| 模型下载失败 | 使用离线缓存模型路径；或切换到 `gpt2`（英文版，更小） |
| bitsandbytes 不可用 | 跳过 Cell 32 的 QLoRA demo，用 Cell 30-31 的理论 + 可视化替代 |
| 训练时间过长 | 减少 epochs 数（3 即可看到趋势）；减少训练数据量 |
| Cell 19 图表不显示 | 检查 matplotlib 安装；用 `%matplotlib inline` |